# FL Participation × Non-IID Experimental Study

This notebook runs all 48 experimental configurations:
- 3 algorithms (FedAvg, FedProx, SCAFFOLD)
- 4 Dirichlet α values (0.05, 0.1, 0.5, 1.0)
- 4 participation rates (20%, 50%, 80%, 100%)

Results are saved incrementally to Google Drive (crash-safe).

**Estimated time:** ~2-3 hours on T4 GPU, ~4 hours on CPU.

In [ ]:
# Cell 1: Mount Drive and Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/FL_Paper_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)

!pip install torch torchvision numpy pandas matplotlib seaborn -q
print('Installation complete')

In [ ]:
# Cell 2: Imports and Configuration
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset
import json, csv, time
from datetime import datetime

# ── Experiment grid ──────────────────────────────────────────────
ALPHAS         = [0.05, 0.1, 0.5, 1.0]
PART_RATES     = [0.2, 0.5, 0.8, 1.0]
ALGORITHMS     = ['fedavg', 'fedprox', 'scaffold']
NUM_CLIENTS    = 10
NUM_ROUNDS     = 30
LOCAL_EPOCHS   = 1
BATCH_SIZE     = 256
LR             = 0.01
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_FILE   = f'{RESULTS_DIR}/all_results.csv'

print(f'Device: {DEVICE}')
print(f'Total experiments: {len(ALPHAS)*len(PART_RATES)*len(ALGORITHMS)}')

In [ ]:
# Cell 3: Model Definition — LiteCNN (~52K params)
class LiteCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

def get_weights(model):
    return [val.cpu().numpy() for _, val in model.state_dict().items()]

def set_weights(model, weights):
    params_dict = zip(model.state_dict().keys(), weights)
    state_dict = {k: torch.tensor(v) for k, v in params_dict}
    model.load_state_dict(state_dict, strict=True)

def evaluate_model(model, loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            loss_sum += criterion(out, y).item()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct/total, loss_sum/len(loader)

print(f'LiteCNN params: {sum(p.numel() for p in LiteCNN().parameters()):,}')

In [ ]:
# Cell 4: Dirichlet Data Partitioner + Load MNIST
def dirichlet_partition(dataset, num_clients, alpha, seed=42):
    np.random.seed(seed)
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    num_classes = len(np.unique(labels))
    class_indices = [np.where(labels == c)[0] for c in range(num_classes)]
    client_indices = [[] for _ in range(num_clients)]
    
    for c_idx in class_indices:
        np.random.shuffle(c_idx)
        proportions = np.random.dirichlet([alpha] * num_clients)
        proportions = (proportions * len(c_idx)).astype(int)
        proportions[-1] = len(c_idx) - proportions[:-1].sum()
        start = 0
        for client_id, count in enumerate(proportions):
            client_indices[client_id].extend(c_idx[start:start+count].tolist())
            start += count
    return client_indices

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
trainset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=256, shuffle=False)
print(f'Data loaded. Train={len(trainset)}, Test={len(testset)}')

In [ ]:
# Cell 5: FL Algorithm Local Training Functions

def local_train_fedavg(model, loader, epochs, lr):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(X), y).backward()
            optimizer.step()
    return get_weights(model)

def local_train_fedprox(model, loader, epochs, lr, global_weights, mu=0.01):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    global_params = [torch.tensor(w).to(DEVICE) for w in global_weights]
    for _ in range(epochs):
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            prox = sum(((p - gp)**2).sum() for p, gp in zip(model.parameters(), global_params))
            (loss + (mu/2) * prox).backward()
            optimizer.step()
    return get_weights(model)

def local_train_scaffold(model, loader, epochs, lr, global_weights, c_global, c_local):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr)  # No momentum for SCAFFOLD
    criterion = nn.CrossEntropyLoss()
    c_g = [torch.tensor(c).to(DEVICE) for c in c_global]
    c_l = [torch.tensor(c).to(DEVICE) for c in c_local]
    K = epochs * len(loader)
    
    for _ in range(epochs):
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(X), y).backward()
            with torch.no_grad():
                for param, cg, cl in zip(model.parameters(), c_g, c_l):
                    if param.grad is not None:
                        param.grad += cg - cl
            optimizer.step()
    
    new_weights = get_weights(model)
    new_c_local = [
        cl.cpu().numpy() - cg.cpu().numpy() + (gw - nw) / (K * lr)
        for cl, cg, gw, nw in zip(c_l, c_g, global_weights, new_weights)
    ]
    delta_c = [nc - oc.cpu().numpy() for nc, oc in zip(new_c_local, c_l)]
    return new_weights, new_c_local, delta_c

print('Client functions defined')

In [ ]:
# Cell 6: Main Experiment Runner
def run_experiment(algorithm, alpha, participation_rate):
    exp_id = f'{algorithm}_alpha{alpha}_part{participation_rate}'
    
    # Resume support: skip completed experiments
    if os.path.exists(RESULTS_FILE):
        existing = pd.read_csv(RESULTS_FILE)
        if exp_id in existing['exp_id'].values:
            print(f'  SKIP (already done): {exp_id}')
            return None
    
    t0 = time.time()
    
    np.random.seed(42)
    client_data_indices = dirichlet_partition(trainset, NUM_CLIENTS, alpha)
    client_loaders = [
        DataLoader(Subset(trainset, indices), batch_size=BATCH_SIZE, shuffle=True)
        for indices in client_data_indices
    ]
    
    global_model = LiteCNN().to(DEVICE)
    global_weights = get_weights(global_model)
    
    c_global = [np.zeros_like(w) for w in global_weights]
    c_locals = [[np.zeros_like(w) for w in global_weights] for _ in range(NUM_CLIENTS)]
    
    num_selected = max(1, int(NUM_CLIENTS * participation_rate))
    round_results = []
    rounds_to_80 = None
    
    for rnd in range(NUM_ROUNDS):
        np.random.seed(42 + rnd)
        selected = np.random.choice(NUM_CLIENTS, num_selected, replace=False)
        local_weights_list = []
        
        for cid in selected:
            local_model = LiteCNN().to(DEVICE)
            set_weights(local_model, global_weights)
            
            if algorithm == 'fedavg':
                lw = local_train_fedavg(local_model, client_loaders[cid], LOCAL_EPOCHS, LR)
                local_weights_list.append(lw)
            elif algorithm == 'fedprox':
                lw = local_train_fedprox(local_model, client_loaders[cid], LOCAL_EPOCHS, LR, global_weights)
                local_weights_list.append(lw)
            elif algorithm == 'scaffold':
                lw, new_cl, delta_c = local_train_scaffold(
                    local_model, client_loaders[cid], LOCAL_EPOCHS, LR,
                    global_weights, c_global, c_locals[cid]
                )
                local_weights_list.append(lw)
                c_locals[cid] = new_cl
                c_global = [cg + (dc / NUM_CLIENTS) for cg, dc in zip(c_global, delta_c)]
        
        # Weighted aggregation
        sizes = [len(client_data_indices[i]) for i in selected]
        total = sum(sizes)
        new_weights = [
            sum(w[layer] * s / total for w, s in zip(local_weights_list, sizes))
            for layer in range(len(global_weights))
        ]
        global_weights = new_weights
        set_weights(global_model, global_weights)
        
        # Evaluate every 3 rounds and at round 1
        if (rnd + 1) % 3 == 0 or rnd == 0:
            acc, loss = evaluate_model(global_model, testloader)
            if acc >= 0.80 and rounds_to_80 is None:
                rounds_to_80 = rnd + 1
            round_results.append({'round': rnd+1, 'accuracy': acc, 'loss': loss})
    
    # Per-client accuracy for fairness analysis
    per_client_accs = []
    for cid in range(NUM_CLIENTS):
        local_model = LiteCNN().to(DEVICE)
        set_weights(local_model, global_weights)
        acc, _ = evaluate_model(local_model, DataLoader(
            Subset(trainset, client_data_indices[cid]), batch_size=256))
        per_client_accs.append(acc)
    
    final_acc = round_results[-1]['accuracy']
    elapsed = time.time() - t0
    
    # Save immediately (crash-safe)
    row = {
        'exp_id': exp_id,
        'algorithm': algorithm,
        'alpha': alpha,
        'participation_rate': participation_rate,
        'final_accuracy': final_acc,
        'rounds_to_80pct': rounds_to_80 if rounds_to_80 else 999,
        'per_client_acc_mean': np.mean(per_client_accs),
        'per_client_acc_std': np.std(per_client_accs),
        'per_client_acc_min': np.min(per_client_accs),
        'round_results': json.dumps(round_results)
    }
    
    file_exists = os.path.exists(RESULTS_FILE)
    with open(RESULTS_FILE, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    
    print(f'  {exp_id} → acc={final_acc:.4f}  ({elapsed:.0f}s)')
    return row

print('Experiment runner ready')

In [ ]:
# Cell 7: Run All 48 Experiments
# Auto-skips completed experiments on resume.

t_start = time.time()
results_summary = []

for alg in ALGORITHMS:
    print(f'\n=== {alg.upper()} ===')
    for alpha in ALPHAS:
        for part in PART_RATES:
            result = run_experiment(alg, alpha, part)
            if result:
                results_summary.append(result)

elapsed = (time.time() - t_start) / 60
print(f'\nCompleted {len(results_summary)} new experiments in {elapsed:.1f} min')
print(f'Results saved to: {RESULTS_FILE}')

In [ ]:
# Cell 8: Generate All Publication Figures

df = pd.read_csv(RESULTS_FILE)
df['round_results'] = df['round_results'].apply(json.loads)

# ── Figure 1: Convergence curves (alpha=0.1, all participation rates) ────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, alg in enumerate(ALGORITHMS):
    for part in PART_RATES:
        row = df[(df['algorithm']==alg) & (df['alpha']==0.1) & (df['participation_rate']==part)].iloc[0]
        rr = row['round_results']
        axes[i].plot([r['round'] for r in rr], [r['accuracy'] for r in rr],
                marker='o', markersize=4, label=f'p={part}', linewidth=2)
    axes[i].set_xlabel('Communication Round', fontsize=12)
    axes[i].set_ylabel('Test Accuracy', fontsize=12)
    axes[i].set_title(f'{alg.upper()} (\u03b1=0.1)', fontsize=13)
    axes[i].legend(fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim([0, 1])
plt.suptitle('Convergence Under Severe Non-IID (\u03b1=0.1)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig1_convergence_alpha01.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Accuracy heatmaps (all algorithms side-by-side) ──────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, alg in enumerate(ALGORITHMS):
    pivot = df[df['algorithm']==alg].pivot(
        index='participation_rate', columns='alpha', values='final_accuracy')
    sns.heatmap(pivot, ax=axes[i], annot=True, fmt='.3f',
                cmap='RdYlGn', vmin=0.5, vmax=1.0,
                cbar_kws={'label': 'Final Accuracy'})
    axes[i].set_title(f'{alg.upper()}', fontsize=13)
    axes[i].set_xlabel('Dirichlet \u03b1', fontsize=10)
    axes[i].set_ylabel('Participation Rate', fontsize=10)
plt.suptitle('Final Accuracy: Algorithm \u00d7 Participation Rate \u00d7 Non-IID Severity',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig2_heatmap_all.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Rounds to 80% at 20% participation ──────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
width = 0.25
x = np.arange(len(ALPHAS))
for j, alg in enumerate(ALGORITHMS):
    vals = []
    for a in ALPHAS:
        row = df[(df['algorithm']==alg) & (df['alpha']==a) & (df['participation_rate']==0.2)]
        v = row['rounds_to_80pct'].values[0] if len(row) > 0 else 999
        vals.append(min(v, 30))
    ax.bar(x + j*width, vals, width, label=alg.upper(), alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels([f'\u03b1={a}' for a in ALPHAS])
ax.set_ylabel('Rounds to Reach 80% Accuracy', fontsize=12)
ax.set_title('Convergence Speed Under 20% Participation Rate', fontsize=13)
ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='Max rounds')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig3_rounds_to_80.png', dpi=150)
plt.show()

# ── Figure 4: Degradation at low participation ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, a in zip(axes, [0.05, 0.5]):
    for alg in ALGORITHMS:
        accs = []
        for part in PART_RATES:
            row = df[(df['algorithm']==alg) & (df['alpha']==a) & (df['participation_rate']==part)]
            accs.append(row['final_accuracy'].values[0] if len(row) > 0 else 0)
        ax.plot(PART_RATES, accs, marker='o', label=alg.upper(), linewidth=2)
    ax.set_xlabel('Participation Rate', fontsize=11)
    ax.set_ylabel('Final Accuracy', fontsize=11)
    ax.set_title(f'Accuracy vs Participation (\u03b1={a})', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig4_participation_degradation.png', dpi=150)
plt.show()

# ── Figure 5: Per-client accuracy variance ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, (ax, alpha_val) in enumerate(zip(axes, [0.05, 1.0])):
    x_pos = np.arange(len(PART_RATES))
    width = 0.25
    for j, alg in enumerate(ALGORITHMS):
        means, stds = [], []
        for part in PART_RATES:
            row = df[(df['algorithm']==alg) & (df['alpha']==alpha_val) & (df['participation_rate']==part)]
            if len(row) > 0:
                means.append(row['per_client_acc_mean'].values[0])
                stds.append(row['per_client_acc_std'].values[0])
            else:
                means.append(0)
                stds.append(0)
        ax.bar(x_pos + j*width, means, width, yerr=stds, capsize=3,
               label=alg.upper(), alpha=0.85)
    ax.set_xticks(x_pos + width)
    ax.set_xticklabels([f'p={p}' for p in PART_RATES])
    ax.set_ylabel('Per-Client Accuracy (mean \u00b1 std)', fontsize=11)
    ax.set_title(f'Client Fairness (\u03b1={alpha_val})', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig5_per_client_variance.png', dpi=150)
plt.show()

# ── Summary Table ─────────────────────────────────────────────────────
print('\n' + '='*60)
print('SUMMARY TABLE — Best algorithm per (α, participation) cell:')
print('='*60)
print(f'{"α":>6} {"p":>6} {"Best":>10} {"Acc%":>8}')
print('-'*36)
for a in ALPHAS:
    for p in PART_RATES:
        subset = df[(df['alpha']==a) & (df['participation_rate']==p)]
        best = subset.loc[subset['final_accuracy'].idxmax()]
        print(f'{a:6.2f} {p:6.1f} {best["algorithm"]:>10} {best["final_accuracy"]*100:8.2f}')

print(f'\nAll figures saved to: {RESULTS_DIR}')